# 06 - Strands Agent
Define a Strands agent with custom @tool decorators, BedrockModel, and run it locally with graph and KB tools.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
MODEL_ID = config["MODEL_ID"]
GRAPH_ID = config["NEPTUNE_GRAPH_ID"]
KB_ID = config["KB_ID"]
GUARDRAIL_ID = config["GUARDRAIL_ID"]
GUARDRAIL_VERSION = config["GUARDRAIL_VERSION"]

tracker = ResourceTracker()
print(f"\u2713 Config loaded (model={MODEL_ID})")

## Step 2: Define Custom Tools

Create @tool-decorated functions that the agent can call.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

neptune = boto3.client("neptune-graph", region_name=REGION)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)


@tool
def search_experts(topic: str, limit: int = 3) -> str:
    """Search the knowledge graph for experts on a given topic.
    
    Args:
        topic: The topic to search for experts in
        limit: Maximum number of results to return
    """
    query = """
    MATCH (p:Person)-[r:KNOWS_ABOUT]->(t:Topic)
    WHERE toLower(t.name) CONTAINS toLower($topic)
    RETURN p.name AS name, p.title AS title, t.name AS topic, r.strength AS strength
    ORDER BY r.strength DESC
    LIMIT $limit
    """
    response = neptune.execute_query(
        graphIdentifier=GRAPH_ID,
        queryString=query,
        language="OPEN_CYPHER",
        parameters={
            "topic": {"value": topic},
            "limit": {"value": limit}
        }
    )
    results = json.loads(response["payload"].read())
    if not results["results"]:
        return f"No experts found for topic: {topic}"
    
    output = []
    for row in results["results"]:
        output.append(f"- {row['name']} ({row['title']}): {row['topic']} (strength: {row['strength']})")
    return "\n".join(output)


print("\u2713 search_experts tool defined")

In [ ]:
@tool
def search_documents(query: str, num_results: int = 3) -> str:
    """Search the knowledge base for relevant documents using RAG retrieval.
    
    Args:
        query: Natural language search query
        num_results: Number of results to return
    """
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {"numberOfResults": num_results}
        }
    )
    
    if not response["retrievalResults"]:
        return "No relevant documents found."
    
    output = []
    for i, result in enumerate(response["retrievalResults"], 1):
        score = result.get("score", 0)
        text = result["content"]["text"][:300]
        output.append(f"[{i}] (score: {score:.3f}) {text}")
    return "\n\n".join(output)


print("\u2713 search_documents tool defined")

In [ ]:
@tool
def get_person_profile(name: str) -> str:
    """Get a person's full profile including expertise areas and authored documents.
    
    Args:
        name: The person's name to look up
    """
    query = """
    MATCH (p:Person)
    WHERE toLower(p.name) CONTAINS toLower($name)
    OPTIONAL MATCH (p)-[r:KNOWS_ABOUT]->(t:Topic)
    OPTIONAL MATCH (p)-[:AUTHORED]->(d:Document)
    RETURN p.name AS name, p.title AS title, p.email AS email,
           collect(DISTINCT {topic: t.name, strength: r.strength}) AS expertise,
           collect(DISTINCT d.title) AS documents
    """
    response = neptune.execute_query(
        graphIdentifier=GRAPH_ID,
        queryString=query,
        language="OPEN_CYPHER",
        parameters={"name": {"value": name}}
    )
    results = json.loads(response["payload"].read())
    
    if not results["results"]:
        return f"No person found matching: {name}"
    
    person = results["results"][0]
    output = f"Name: {person['name']}\nTitle: {person['title']}\nEmail: {person['email']}\n"
    output += "\nExpertise:\n"
    for exp in person["expertise"]:
        if exp["topic"]:
            output += f"  - {exp['topic']} (strength: {exp['strength']})\n"
    output += "\nAuthored Documents:\n"
    for doc in person["documents"]:
        if doc:
            output += f"  - {doc}\n"
    return output


print("\u2713 get_person_profile tool defined")

## Step 3: Create the Strands Agent

Wire up BedrockModel with our tools and system prompt.

In [ ]:
# Create the Bedrock model
model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION
)

# Define the system prompt
SYSTEM_PROMPT = """You are OrgIQ Search, an organizational intelligence assistant.
You help users find experts, documents, and knowledge within the organization.

You have access to:
- A knowledge graph with people, topics, and documents
- A document knowledge base with full-text search

When answering questions:
1. Use search_experts to find people with relevant expertise
2. Use search_documents to find relevant documentation
3. Use get_person_profile for detailed information about specific people
4. Synthesize results into clear, actionable answers
5. Always cite your sources (person names, document titles)
"""

# Create the agent
agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_experts, search_documents, get_person_profile]
)

print("\u2713 Strands Agent created")
print(f"  Model: {MODEL_ID}")
print(f"  Tools: search_experts, search_documents, get_person_profile")

## Step 4: Test - Expert Search Query

In [ ]:
# Test the agent with an expert search query
response = agent("Who are the best Kubernetes experts in our organization?")
print("Agent response:")
print(response)

## Step 5: Test - Document Search Query

In [ ]:
# Test document search
response = agent("What do we know about zero trust authentication? Summarize our documentation.")
print("Agent response:")
print(response)

## Step 6: Test - Person Profile Query

In [ ]:
# Test person profile lookup
response = agent("Tell me about Alice Chen - what does she work on and what has she written?")
print("Agent response:")
print(response)

## Step 7: Test - Multi-Tool Complex Query

In [ ]:
# Complex query requiring multiple tools
response = agent(
    "I need to set up a new ML pipeline. Who should I talk to, "
    "and what existing documentation can help me get started?"
)
print("Agent response:")
print(response)

## Step 8: Agent with Guardrails

Create a version of the agent with guardrails applied.

In [ ]:
# Create a guarded model
guarded_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    guardrail_config={
        "guardrailIdentifier": GUARDRAIL_ID,
        "guardrailVersion": GUARDRAIL_VERSION
    }
)

guarded_agent = Agent(
    model=guarded_model,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_experts, search_documents, get_person_profile]
)

print("\u2713 Guarded agent created")

# Test with safe query
print("\n--- Safe query ---")
response = guarded_agent("Who knows about CI/CD?")
print(response)

In [ ]:
# Test with query that should be blocked
print("--- Blocked query (salary) ---")
try:
    response = guarded_agent("What is Alice's salary?")
    print(response)
except Exception as e:
    print(f"\u2713 Guardrail blocked the request: {e}")

## Step 9: Streaming Agent Response

In [ ]:
# Use streaming for real-time output
import sys

print("Streaming response:")
print("-" * 40)

result = agent.stream("Give me a brief overview of our security expertise.")
for event in result:
    if hasattr(event, "data"):
        sys.stdout.write(event.data)
        sys.stdout.flush()

print("\n" + "-" * 40)
print("\u2713 Streaming complete")

## Step 10: Agent Conversation History

In [ ]:
# Multi-turn conversation
conversation_agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[search_experts, search_documents, get_person_profile]
)

# Turn 1
print("User: Who knows about authentication?")
response1 = conversation_agent("Who knows about authentication?")
print(f"Agent: {response1}\n")

# Turn 2 - referencing previous context
print("User: What documents have they written?")
response2 = conversation_agent("What documents have they written?")
print(f"Agent: {response2}\n")

print("\u2713 Multi-turn conversation working")

## Summary

- [x] Three custom @tool functions defined (search_experts, search_documents, get_person_profile)
- [x] Strands Agent created with BedrockModel (Claude Sonnet)
- [x] Expert search queries working via Neptune graph
- [x] Document search working via Bedrock KB RAG
- [x] Person profile lookups working
- [x] Multi-tool complex queries handled
- [x] Guardrail integration verified (safe pass, unsafe block)
- [x] Streaming responses working
- [x] Multi-turn conversation context maintained

**Next:** [07-agentcore-memory.ipynb](07-agentcore-memory.ipynb) — Add persistent memory to the agent with AgentCore Memory